### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from astropy.coordinates import SkyCoord, CartesianRepresentation
import astropy.units as u
from astropy.table import Table
from astropy.visualization.wcsaxes.frame import EllipticalFrame
from astropy.wcs import WCS
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import patheffects as pe
import matplotlib.pyplot as plt
from mocpy import MOC
import numpy as np
from regions import Regions
import aomap
from survey_tools import healpix
from ao_tools import etc, simulate, training

### Options

In [ ]:
config = aomap.read_config('config.yaml')

ao_system_name = 'GNAO'
map_level = 6

ao_system = next((system for system in config.ao_systems if system['name'] == ao_system_name), None)

extragalactic_3d_hst_fields = np.array([
    [214.65  ,  52.65  , 'AEGIS'],
    [150.1292,   2.40  , 'COSMOS'],
    [188.9790,  62.1976, 'GOODS-N'],
    [ 53.1250, -27.7886, 'GOODS-S'],
    [ 34.4541,  -5.2006, 'UDS']
])

In [ ]:
fontsize_big = {
    'xlabel': 20,
    'ylabel': 20,
    'title': 24,
    'xtick_label': 16,
    'ytick_label': 16,
    'cbar_label': 20,
    'cbar_tick_label': 16,
    'points_label': 14,
    'stars_legend': 16
}

fontsize = {
    'xlabel': 16,
    'ylabel': 16,
    'title': 20,
    'xtick_label': 12,
    'ytick_label': 12,
    'cbar_label': 16,
    'cbar_tick_label': 12,
    'points_label': 10,
    'stars_legend': 12
}

red_highlight_color = '#D0343E'
darkred_highlight_color = '#810001'
tan_highlight_color = '#D6D0CA'
blue_highlight_color = '#3AC2EF'

colors = {
    'badvalue': '#6e6e6e',
    'milkyway': '#4a4a4a'
}

alphas = {
    'grid': 0.4
}

outline_effect = [pe.Stroke(linewidth=2.8, foreground='black', alpha=0.6), pe.Normal()]

In [ ]:
print(f"Map Level: {map_level}")
print(f"Map Resolution: {healpix.get_resolution(map_level).to(u.deg):.2f}")
print(f"Map Area: {healpix.get_area(map_level).to(u.deg**2):.2f}")
print()
print(f"Outer Level: {config.outer_level}")
print(f"Outer Resolution: {healpix.get_resolution(config.outer_level).to(u.deg):.2f}")
print(f"Outer Area: {healpix.get_area(config.outer_level).to(u.deg**2):.2f}")
print()
print(f"Inner Level: {config.inner_level}")
print(f"Inner Resolution: {healpix.get_resolution(config.inner_level).to(u.arcsec):.2f}")
print(f"Inner Area: {healpix.get_area(config.inner_level).to(u.arcsec**2):.2f}")

### Star Density

In [ ]:
aomap.plot_map(aomap.get_map_data(config, map_level, 'star-density'), galactic=True, 
               width=12, dpi=150, grid=False, norm='symlog', cmap='gray', cbar_format='%g',
               fontsize=fontsize_big, hide_title=True)

### Plot Asterims

In [ ]:

# Cosmos
level = 8
pix = 436131 # cosmos
#pix = 245755 # edfn
outer_pix = healpix.get_parent_pixel(level, pix, config.outer_level)

print(f"Outer Pixel: {outer_pix}")

asterisms = aomap.load_asterisms(config, outer_pix, ao_system_name)
stars = aomap.get_stars_for_asterisms(config, outer_pix, required_band=ao_system['band'], use_cache=True)
print(f"Total Asterisms in Outer Pixel: {len(asterisms):,.0f}")
print(f"Total Stars in Outer Pixel: {len(stars):,.0f}")

print(f"Zoom Level {level} Pixel: {pix}")

plot_level = config.inner_level
min_mag = int(np.floor(ao_system['nom_mag']))
max_mag = int(np.ceil(ao_system['max_mag'])) if ao_system['nom_mag'] < ao_system['max_mag'] else 19

aomap.plot_map(aomap.get_dummy_map_data(config, plot_level, level=level, pixs=pix),
    stars=[stars, {'band': ao_system['band'], 'min_mag': min_mag, 'max_mag': max_mag, 'max_size': 50, 'fc': 'white', 'ec': 'black'}],
    asterisms=[asterisms, {'fov': ao_system['fov']}],
    projection='astro', norm='linear', width=12, boundaries_level=level, boundaries_pixs=0,
    colors={'badvalue': 'white'}, fontsize=fontsize_big, hide_title=True, hide_cbar=True
)

### Asterism Coverage

In [ ]:
aomap.plot_map(aomap.get_map_data(config, map_level, f"asterism-coverage-{ao_system_name}", nan_below=0.01),
               milkyway=True, milkyway_width=20,
               rotation=110, projection='cart', width=15, vmin=0.0, vmax=1.0,
               fontsize=fontsize, alphas=alphas, colors=colors,
               hide_title=True
)

In [ ]:
aomap.plot_map(aomap.get_map_data(config, map_level, f"asterism-coverage-{ao_system_name}", survey='ews', nan_below=0.01),
               milkyway=True, milkyway_width=20,
               surveys=[['ews', {'edgecolor': red_highlight_color, 'linewidth': 1.5, 'linestyle': 'solid'}],
                        ['edf-north', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-N', [-12,8], {'path_effects': outline_effect}],
                        ['edf-south', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-S', [-10,6], {'path_effects': outline_effect}],
                        ['edf-fornax', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-F', [-10,6], {'path_effects': outline_effect}]],
               points=[[extragalactic_3d_hst_fields, {'marker': 's', 'edgecolor': 'white', 'facecolor': 'none', 'zorder': 2}, {'path_effects': outline_effect}]],
               rotation=110, projection='cart', width=15, vmin=0.0, vmax=1.0,
               fontsize=fontsize, colors=colors, alphas=alphas,
               hide_title=True
)

In [ ]:
aomap.plot_map(aomap.get_map_data(config, map_level, f"asterism-coverage-{ao_system_name}", nan_below=0.01),
               milkyway=True, milkyway_width=20, galactic=True,
               surveys=[['ews', {'edgecolor': 'red', 'linewidth': 1.5, 'linestyle': 'solid'}],
                        ['edf-north', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-N', [-12,8], {'path_effects': outline_effect}],
                        ['edf-south', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-S', [-12,9], {'path_effects': outline_effect}],
                        ['edf-fornax', {'edgecolor': '#B5EFFF', 'linewidth': 1.5, 'linestyle': 'solid'}, 'EDF-F', [-12,7], {'path_effects': outline_effect}]],
               points=[[extragalactic_3d_hst_fields, {'marker': 's', 'edgecolor': 'white', 'facecolor': 'none'}, {'path_effects': outline_effect}]],
               projection='cart', width=15, vmax=1.0,
               fontsize=fontsize, colors=colors, alphas=alphas,
               hide_title=True
)

### AO Friendly Areas

In [ ]:
coverage_data = aomap.get_map_data(config, map_level, f"asterism-coverage-{ao_system_name}", nan_below=0.01)
dust_data = aomap.get_map_data(config, map_level, "dust-extinction")

total_pix = len(coverage_data.values)
galactic_coords = coverage_data.coords.galactic
ecliptic_coords = coverage_data.coords.barycentricmeanecliptic

mean_coverage_pole = np.mean(coverage_data.values[(coverage_data.values > 0.0) & (np.abs(galactic_coords.b.deg) > 75.0)])
print(f"Mean Coverage Near Galactic Poles: {mean_coverage_pole:.3f}")

In [ ]:
cmap = LinearSegmentedColormap.from_list("ao_coverage", ["#ff6666", "#ff0000", red_highlight_color])

In [ ]:
for min_coverage in [0.8, 0.85, 0.87, 0.9]:
    pixs_filter = (coverage_data.values >= min_coverage) & (np.abs(galactic_coords.b.deg) > 20.0) & (np.abs(ecliptic_coords.lat.deg) > 15.0) & (dust_data.values < 0.3)

    num_pix = np.sum(pixs_filter)
    print(f'Fraction of Pix with Coverage ≥ {min_coverage:.0%}: {num_pix/total_pix:.1%} ({num_pix}/{total_pix}), Area: {num_pix*healpix.get_area(map_level).to(u.deg**2):.1f}')

    aomap.plot_map(aomap.mask_map_data(coverage_data, pixs_filter),
                milkyway=True, milkyway_width=20,
                surveys=[['ews', {'edgecolor': 'white', 'linewidth': 0.5, 'linestyle': 'solid'}]],
                rotation=110, projection='cart', width=15, cmap=cmap, vmin=0.8, vmax=1.0,
                fontsize=fontsize, colors=colors, alphas=alphas,
                hide_title=True
    )

In [ ]:
min_coverage = 0.9
dec_limit = [-20,15]
best_pixs_filter  = (coverage_data.values >= min_coverage) & (np.abs(galactic_coords.b.deg) > 20.0) & (np.abs(ecliptic_coords.lat.deg) > 15.0) & (dust_data.values < 0.3)
best_pixs_filter &= (coverage_data.coords.dec.deg >= dec_limit[0]) & (coverage_data.coords.dec.deg <= dec_limit[1])

best_pixs_points = []
num_pix = np.sum(best_pixs_filter)
print(f'Fraction of Pix with Coverage ≥ {min_coverage:.0%}: {num_pix/total_pix:.1%} ({num_pix}/{total_pix}), Area: {num_pix*healpix.get_area(map_level).to(u.deg**2):.1f}')
best_pixs_idx = np.where(best_pixs_filter)[0]
for i in range(len(best_pixs_idx)):
    pix = best_pixs_idx[i]
    coord = coverage_data.coords[pix]
    best_pixs_points.append([coord.ra.deg, coord.dec.deg])
    print(f"  - {pix}: RA={coord.ra.deg:.3f}, Dec={coord.dec.deg:.3f}, l={coord.galactic.l.deg:.3f}, b={coord.galactic.b.deg:.3f}, coverage={coverage_data.values[pix]:.3f}, dust={dust_data.values[pix]:.3f}")
aomap.plot_map(aomap.mask_map_data(coverage_data, best_pixs_filter), zoom=False,
               milkyway=True, milkyway_width=20,
               lines=[{'dec': dec_limit[0], 'lw': 2.0, 'color': tan_highlight_color, 'alpha': 0.7}, {'dec': dec_limit[1], 'lw': 2.0, 'color': tan_highlight_color, 'alpha': 0.7}],
               surveys=[['ews', {'edgecolor': 'white', 'linewidth': 1.0, 'linestyle': 'solid'}]],
               points=[[best_pixs_points, {'s': 400, 'marker': 'o', 'edgecolor': blue_highlight_color, 'facecolor': 'none', 'linewidth': 2.5}]],
               rotation=110, projection='cart', width=15, cmap=cmap, vmin=0.8, vmax=1.0,
               fontsize=fontsize, colors=colors, alphas=alphas,
               hide_title=True
)


In [ ]:
pixel_offset = 2
for zoom in range(0,4):
    outer_pix = best_pixs_idx[1]
    print(f"Outer Pixel: {outer_pix}")

    level = config.outer_level + zoom
    pix = outer_pix
    for i in range(zoom):
        level_pix = healpix.get_subpixels(config.outer_level+i, pix, config.outer_level+i+1)
        pix = level_pix[pixel_offset]

    asterisms = aomap.load_asterisms(config, outer_pix, ao_system_name)
    stars = aomap.get_stars_for_asterisms(config, outer_pix, required_band=ao_system['band'], use_cache=True)
    print(f"Total Asterisms in Outer Pixel: {len(asterisms):,.0f}")
    print(f"Total Stars in Outer Pixel: {len(stars):,.0f}")
    print(f"Zoom Level {level} Pixel: {pix}")

    plot_level = config.inner_level
    min_mag = int(np.floor(ao_system['nom_mag']))
    max_mag = int(np.ceil(ao_system['max_mag'])) if ao_system['nom_mag'] < ao_system['max_mag'] else 19

    aomap.plot_map(aomap.get_dummy_map_data(config, plot_level, level=level, pixs=pix),
        stars=[stars, {'band': ao_system['band'], 'min_mag': min_mag, 'max_mag': max_mag, 'max_size': 50, 'fc': 'white', 'ec': 'black'}],
        asterisms=[asterisms, {'fov': ao_system['fov']}],
        projection='astro', norm='linear', width=12, boundaries_level=level, boundaries_pixs=0,
        colors={'badvalue': 'white'}, fontsize=fontsize_big, hide_title=True, hide_cbar=True
    )

### Load Asterisms

In [ ]:
fits_path = '../output/asterisms-2025-07'
fits_file = f"{fits_path}/asterisms-GNAO.fits"
asterisms = Table.read(fits_file, format='fits')
print(f"Number of asterisms: {len(asterisms):,.0f}")

In [ ]:
print(asterisms.colnames)

### Asterism Stats

In [ ]:
def mean_skycoord(coords):
    # Cartesian unit vectors
    xyz = coords.cartesian  # CartesianRepresentation
    x = xyz.x.to_value(u.one)
    y = xyz.y.to_value(u.one)
    z = xyz.z.to_value(u.one)

    vx, vy, vz = x.mean(), y.mean(), z.mean()

    v = np.array([vx, vy, vz], dtype=float)
    n = np.linalg.norm(v)
    if n == 0.0:
        raise ValueError("Mean direction undefined (vectors cancel).")
    v /= n

    rep = CartesianRepresentation(v[0]*u.one, v[1]*u.one, v[2]*u.one)
    return SkyCoord(rep, frame=coords.frame)

def get_asterism_mask(asterisms, filenames, levels=14):
    if not isinstance(filenames, list):
        filenames = [filenames]

    map_centres = []
    moc = None
    for filename in filenames:
        if filename.endswith('reg'):
            regions = Regions.read(filename, format="ds9")
            region_moc = MOC.from_astropy_regions(regions, max_depth=levels)
        else:
            region_moc = MOC.from_fits(filename)

        region_pixs = region_moc.flatten().astype(int)
        region_coords = healpix.get_pixel_skycoord(region_moc.max_order, region_pixs)
        region_centre = mean_skycoord(region_coords)
        map_centres.append(region_centre)

        if moc is None:
            moc = region_moc
        else:
            moc |= region_moc

    level = moc.max_order
    map_pixs = moc.flatten().astype(int)

    asterism_coords = SkyCoord(ra=asterisms['ra']*u.deg, dec=asterisms['dec']*u.deg)
    asterism_pixs = healpix.get_healpix_from_skycoord(level, asterism_coords)
    mask = np.isin(asterism_pixs, map_pixs)

    ALL_SKY = 4 * np.pi * u.steradian
    map_area = moc.sky_fraction * ALL_SKY

    asterism_moc = MOC.from_cones(
        lon=asterisms['ra'][mask] * u.deg,
        lat=asterisms['dec'][mask] * u.deg,
        radius=1 * u.arcmin,
        max_depth=levels,
        union_strategy="small_cones"
    )

    coverage = asterism_moc.sky_fraction / moc.sky_fraction

    return mask, map_area, coverage, moc, map_centres if len(map_centres) > 1 else map_centres[0], asterism_moc

In [ ]:
map_data = aomap.get_map_data(config, map_level, f"asterism-count-{ao_system_name}")
total_asterisms = np.sum(map_data.values)
print(f'Total Asterisms: {total_asterisms:,.0f}')

mask_with_stars = map_data.values > 0
num_pix_with_stars = np.sum(mask_with_stars)
area_with_stars = num_pix_with_stars * healpix.get_area(map_level)
print(f'Sky Area with Asterisms: {area_with_stars.to(u.deg**2):.2f}')

print(f"Asterism Density: {total_asterisms/area_with_stars.to_value(u.deg**2):.0f} asterisms/deg²")

map_data = aomap.get_map_data(config, map_level, f"asterism-coverage-{ao_system_name}")
mean_coverage = np.mean(map_data.values[mask_with_stars])
print(f'Average Coverage: {mean_coverage:.1%}')


In [ ]:
mask_ews, area_ews, coverage_ews, moc_ews, _, _ = get_asterism_mask(asterisms, "../data/euclid/rsd2024a-footprint-equ-13-year6-MOC.fits", levels=16)
print(f"EWS asterisms: {np.sum(mask_ews):,.0f} in {area_ews.to(u.deg**2):.2f}, ({np.sum(mask_ews)/area_ews.to_value(u.deg**2):.0f} asterisms/deg², {coverage_ews*100:.1f}% coverage)")

In [ ]:
mask_edfn, area_edfn, coverage_edfn, moc_edfn, coord_edfn, asterism_moc_edfn = get_asterism_mask(asterisms, "../data/euclid/EuclidMOC_EDFN_rsd2024c_depth13_atLeast2visitsPlanned.fits", levels=20)
print(f"EDFN asterisms: {np.sum(mask_edfn):,.0f} in {area_edfn.to(u.deg**2):.2f}, ({np.sum(mask_edfn)/area_edfn.to_value(u.deg**2):.0f} asterisms/deg², {coverage_edfn*100:.1f}% coverage, centered at RA={coord_edfn.ra:.1f}, Dec={coord_edfn.dec:.1f}, b={coord_edfn.galactic.b:.1f})")

mask_edfs, area_edfs, coverage_edfs, moc_edfs, coord_edfs, _ = get_asterism_mask(asterisms, "../data/euclid/EuclidMOC_EDFS_rsd2024c_depth13_atLeast2visitsPlanned.fits", levels=20)
print(f"EDFS asterisms: {np.sum(mask_edfs):,.0f} in {area_edfs.to(u.deg**2):.2f}, ({np.sum(mask_edfs)/area_edfs.to_value(u.deg**2):.0f} asterisms/deg², {coverage_edfs*100:.1f}% coverage, centered at RA={coord_edfs.ra:.1f}, Dec={coord_edfs.dec:.1f}, b={coord_edfs.galactic.b:.1f})")

mask_edff, area_edff, coverage_edff, moc_edff, coord_edff, _ = get_asterism_mask(asterisms, "../data/euclid/EuclidMOC_EDFF_rsd2024c_depth13_atLeast2visitsPlanned.fits", levels=20)
print(f"EDFF asterisms: {np.sum(mask_edff):,.0f} in {area_edff.to(u.deg**2):.2f}, ({np.sum(mask_edff)/area_edff.to_value(u.deg**2):.0f} asterisms/deg², {coverage_edff*100:.1f}% coverage, centered at RA={coord_edff.ra:.1f}, Dec={coord_edff.dec:.1f}, b={coord_edff.galactic.b:.1f})")

In [ ]:
mask_cosmos, area_cosmos, coverage_cosmos, moc_cosmos, coord_cosmos, asterism_moc_cosmos = get_asterism_mask(asterisms, "../data/moc/CDS_II_284_cosmos.fits", levels=20)
print(f"COSMOS asterisms: {np.sum(mask_cosmos):,.0f} in {area_cosmos.to(u.deg**2):.2f}, ({np.sum(mask_cosmos)/area_cosmos.to_value(u.deg**2):.0f} asterisms/deg², {coverage_cosmos*100:.1f}% coverage, centered at RA={coord_cosmos.ra:.1f}, Dec={coord_cosmos.dec:.1f}, b={coord_cosmos.galactic.b:.1f})")

mask_3dhst, area_3dhst, coverage_3dhst, moc_3dhst, coords_3dhst, _ = get_asterism_mask(asterisms,["../data/moc/AEGIS_wfc3.3dhst.reg", "../data/moc/COSMOS_wfc3.3dhst.reg", "../data/moc/GOODS-N_wfc3.3dhst_fix.reg", "../data/moc/GOODS-S_wfc3.3dhst.reg", "../data/moc/UDS_wfc3.3dhst_fix.reg"], levels=20)
print(f"3D-HST asterisms: {np.sum(mask_3dhst):,.0f} in {area_3dhst.to(u.deg**2):.2f}, ({np.sum(mask_3dhst)/area_3dhst.to_value(u.deg**2):.0f} asterisms/deg², {coverage_3dhst*100:.1f}% coverage, b={', '.join([f'{c.galactic.b:.1f}' for c in coords_3dhst])})")

In [ ]:
def plot_moc(moc, title):
    fig = plt.figure(111, figsize=(9, 6))
    wcs = WCS(
        {
            "naxis": 2,
            "naxis1": 1620,
            "naxis2": 810,
            "crpix1": 810.5,
            "crpix2": 405.5,
            "cdelt1": -0.2,
            "cdelt2": 0.2,
            "ctype1": "RA---AIT",
            "ctype2": "DEC--AIT",
        },
    )
    ax = fig.add_subplot(1, 1, 1, projection=wcs, frame_class=EllipticalFrame)
    moc.fill(ax=ax, wcs=wcs, alpha=0.5, fill=True, color="red")
    moc.border(ax=ax, wcs=wcs, alpha=0.5, color="black")
    ax.set_aspect(1.0)
    plt.xlabel("ra")
    plt.ylabel("dec")
    plt.title(title)
    plt.grid(color="black", linestyle="dotted")
    plt.show()

plot_moc(moc_edfn, "Euclid Deep Field North (EDFN)")
plot_moc(moc_edfs, "Euclid Deep Field South (EDFS)")
plot_moc(moc_edff, "Euclid Deep Field Fornax (EDFF)")
plot_moc(moc_cosmos, "COSMOS")
plot_moc(moc_3dhst, "3D-HST Fields")

In [ ]:
colors = {'Best':'#D55E00','Mean':'#009E73','Worst':'#0072B2'}
alpha = 0.6

def make_bins(width, *arrays):
    lo = min(np.min(a) for a in arrays)
    hi = max(np.max(a) for a in arrays)
    return np.arange(lo, hi + width, width)


In [ ]:
sr_bins   = make_bins(0.005, asterisms['SR_mean'])
ee_bins   = make_bins(0.005, asterisms['EE100_mean'])
fwhm_bins = make_bins(2.0,   asterisms['FWHM_mean'])

plt.figure(figsize=(8, 5))
plt.hist(asterisms['EE100_mean'], bins=ee_bins, color=colors['Mean'], edgecolor='gray', alpha=alpha)
plt.xlabel('Mean EE [100 mas]', fontsize=16)
plt.ylabel('Asterism Count', fontsize=16)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.grid()
plt.title('H-band (1.654μm)', fontsize=20)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

for ax, (mask, label) in zip(axes, [(mask_edfn, 'EDF-N'), (mask_cosmos, 'COSMOS'), (mask_3dhst, '3D-HST')]):
    ax.hist(asterisms['SR_mean'][mask], bins=sr_bins, alpha=alpha, label=label, color=colors['Mean'], edgecolor='gray')
    ax.set_ylabel('Asterism Count', fontsize=16)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel('Strehl Ratio', fontsize=16)
fig.suptitle('H-band (1.654μm)', fontsize=20)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

axes[0].hist(asterisms['SR_mean'], bins=sr_bins, color=colors['Mean'], edgecolor='gray', alpha=alpha)
axes[0].set_xlabel('SR', fontsize=16)
axes[0].set_ylabel('Asterism Count', fontsize=16)
axes[0].tick_params(axis='both', which='major', labelsize=12)
axes[0].grid()

axes[1].hist(asterisms['EE100_mean'], bins=ee_bins, color=colors['Mean'], edgecolor='gray', alpha=alpha)
axes[1].set_xlabel('EE [100 mas]', fontsize=16)
axes[1].tick_params(axis='both', which='major', labelsize=12)
axes[1].grid()

axes[2].hist(asterisms['FWHM_mean'], bins=fwhm_bins, color=colors['Mean'], edgecolor='gray', alpha=alpha)
axes[2].set_xlabel('FWHM [mas]', fontsize=16)
axes[2].tick_params(axis='both', which='major', labelsize=12)
axes[2].grid()

plt.tight_layout()
plt.show()

In [ ]:
sr_bins   = make_bins(0.005, asterisms['SR_min'],    asterisms['SR_mean'],    asterisms['SR_max'])
ee_bins   = make_bins(0.005, asterisms['EE100_min'], asterisms['EE100_mean'], asterisms['EE100_max'])
fwhm_bins = make_bins(2.0,   asterisms['FWHM_min'],  asterisms['FWHM_mean'],  asterisms['FWHM_max'])

plt.figure(figsize=(8, 5))
plt.hist(asterisms['SR_max'],  bins=sr_bins, alpha=alpha, label='Best',  color=colors['Best'],  edgecolor='gray')
plt.hist(asterisms['SR_mean'], bins=sr_bins, alpha=alpha, label='Mean',  color=colors['Mean'],  edgecolor='gray')
plt.hist(asterisms['SR_min'],  bins=sr_bins, alpha=alpha, label='Worst', color=colors['Worst'], edgecolor='gray')
plt.xlabel('Strehl Ratio')
plt.ylabel('Asterism Count')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('H-band (1.654μm)')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

for ax, (mask, name) in zip(
    axes,
    [(mask_edfn, 'EDF-N'), (mask_cosmos, 'COSMOS'), (mask_3dhst, '3D-HST')]
):
    ax.hist(asterisms['SR_max'][mask],  bins=sr_bins, alpha=alpha, label='Best', color=colors['Best'],  edgecolor='gray')
    ax.hist(asterisms['SR_mean'][mask], bins=sr_bins, alpha=alpha, label='Mean', color=colors['Mean'],  edgecolor='gray')
    ax.hist(asterisms['SR_min'][mask],  bins=sr_bins, alpha=alpha, label='Worst', color=colors['Worst'], edgecolor='gray')
    ax.set_ylabel('Asterism Count')
    ax.grid(True, alpha=0.3)
    ax.legend(title=name)

axes[-1].set_xlabel('Strehl Ratio')
fig.suptitle('H-band (1.654μm)')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

axes[0].hist(asterisms['SR_max'],  bins=sr_bins, alpha=alpha, label='Best',  color=colors['Best'],  edgecolor='gray')
axes[0].hist(asterisms['SR_mean'], bins=sr_bins, alpha=alpha, label='Mean',  color=colors['Mean'],  edgecolor='gray')
axes[0].hist(asterisms['SR_min'],  bins=sr_bins, alpha=alpha, label='Worst', color=colors['Worst'], edgecolor='gray')
axes[0].set_xlabel('SR')
axes[0].set_ylabel('Asterism Count')
axes[0].grid(True, alpha=0.3)

axes[1].hist(asterisms['EE100_max'], bins=ee_bins, alpha=alpha, color=colors['Best'],  edgecolor='gray')
axes[1].hist(asterisms['EE100_mean'],bins=ee_bins, alpha=alpha, color=colors['Mean'],  edgecolor='gray')
axes[1].hist(asterisms['EE100_min'], bins=ee_bins, alpha=alpha, color=colors['Worst'], edgecolor='gray')
axes[1].set_xlabel('EE [100 mas]')
axes[1].grid(True, alpha=0.3)

axes[2].hist(asterisms['FWHM_min'],  bins=fwhm_bins, alpha=alpha, label='Best',  color=colors['Best'],  edgecolor='gray')
axes[2].hist(asterisms['FWHM_mean'], bins=fwhm_bins, alpha=alpha, label='Mean',  color=colors['Mean'],  edgecolor='gray')
axes[2].hist(asterisms['FWHM_max'],  bins=fwhm_bins, alpha=alpha, label='Worst', color=colors['Worst'], edgecolor='gray')
axes[2].set_xlabel('FWHM [mas]')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.suptitle('H-band (1.654μm)')
plt.tight_layout()
plt.show()

### AO Performance

In [ ]:
tiptop_path = '../../girmos-aosims/output/tiptop'
model_path = '../data/models'
all_results_test = [simulate.load_asterism_stats(tiptop_path, f"hybrid_moao_atm_median_za_20_lgs_30_opt_11_sci_11_random_6000_vib22_rnd_test_3star")]

In [ ]:
model_name = f"ao_model_rnd_3star_expanded_mean_1024-512"
model = training.load_model(model_path, model_name)

X_test, Y_test = training.get_model_data(all_results_test, options=model['data_options'])
Y_pred_test = training.get_prediction(X_test, model)

yMin = Y_test[:,0].min()
yMax = Y_test[:,0].max()
dy = 0.05
name = "Strehl Ratio"

fig = plt.figure(figsize=(8, 8))
ax = plt.gca()

# Left: Test set
plt.scatter(Y_test[:, 0], Y_pred_test[:, 0], s=5, alpha=0.5)
plt.plot([yMin, yMax], [yMin-dy, yMax-dy], 'k:', lw=1)
plt.plot([yMin, yMax], [yMin   , yMax   ], 'k--', lw=1)
plt.plot([yMin, yMax], [yMin+dy, yMax+dy], 'k:', lw=1)
plt.xlabel(f"TIPTOP {name}", fontsize=22)
plt.ylabel(f"Predicted {name}", fontsize=22)
plt.tick_params(axis='both', labelsize=18)
plt.xlim(yMin, yMax)
plt.ylim(yMin, yMax)
ax.set_aspect('equal', adjustable='box')
plt.grid()

inset_ax = ax.inset_axes([0.6, 0.05, 0.35, 0.35])
inset_ax.hist((Y_test[:,0] - Y_pred_test[:,0])/Y_pred_test[:,0], bins=100, color='gray', alpha=0.7)
inset_ax.axvline(0, linestyle='--', linewidth=1)
inset_ax.set_xlim(-0.1, 0.1)
inset_ax.tick_params(axis='both', which='major', labelsize=16)
inset_ax.set_xticks([-0.1, -0.05, 0.0, 0.05, 0.1])
inset_ax.set_yticks([])
inset_ax.set_title('Relative Differences', fontsize=19.5)
inset_ax.grid(axis='x')

plt.tight_layout()
plt.show()

In [ ]:
model_name = f"ao_model_rnd_3star_1024-512-256"
model = training.load_model(model_path, model_name)

sim_ids = [3641, 643, 134, 1582]
for sim_id in sim_ids:
    #sim_id = int(np.random.uniform(1,len(all_results_test[0]['sr'])))
    results = simulate.load_tiptop_results(tiptop_path, all_results_test[0]['name'], sim_id=sim_id)
    print(f"Sim ID: {sim_id}")
    print(f"Band: {simulate.get_band(results['wavelength'].value)}")
    print(f"NGS: {simulate.get_ngs_string(results['ngs'][0])}")
    results_pred = training.get_prediction_as_results(results, model)
    true_label = f"TIPTOP Simulation ({results['wavelength'].to_value(u.micron)}μm)"
    pred_label = f"NN Prediction ({results['wavelength'].to_value(u.micron)}μm)"
    fig = simulate.plot_fov(
        [results, results_pred],
        labels=[true_label, pred_label],
        plot_value='SR',
        fixed_range=True,
        plot_mags=False, hide_stats=True,
        return_figure=True
    )
    for ax in fig.axes:
        ax.xaxis.label.set_fontsize(16)
        ax.yaxis.label.set_fontsize(16)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.title.set_fontsize(20)
        ax.title.set_fontweight('normal')
    fig.axes[2].set_ylabel('')
    fig.axes[2].yaxis.label.set_visible(False)
    plt.show()


In [ ]:
X_test, Y_test = training.get_model_data(all_results_test, options=model['data_options'])
Y_pred_test = training.get_prediction(X_test, model)

yMin = Y_test[:,0].min()
yMax = Y_test[:,0].max()
dy = 0.05
name = "Strehl Ratio"

fig = plt.figure(figsize=(8, 8))
ax = plt.gca()

# Left: Test set
plt.scatter(Y_test[:, 0], Y_pred_test[:, 0], s=5, alpha=0.5)
plt.plot([yMin, yMax], [yMin-dy, yMax-dy], 'k:', lw=1)
plt.plot([yMin, yMax], [yMin   , yMax   ], 'k--', lw=1)
plt.plot([yMin, yMax], [yMin+dy, yMax+dy], 'k:', lw=1)
plt.xlabel(f"TIPTOP {name}", fontsize=22)
plt.ylabel(f"Predicted {name}", fontsize=22)
plt.tick_params(axis='both', labelsize=18)
plt.xlim(yMin, yMax)
plt.ylim(yMin, yMax)
ax.set_aspect('equal', adjustable='box')
plt.grid()

inset_ax = ax.inset_axes([0.6, 0.05, 0.35, 0.35])
inset_ax.hist((Y_test[:,0] - Y_pred_test[:,0])/Y_pred_test[:,0], bins=100, color='gray', alpha=0.7)
inset_ax.axvline(0, linestyle='--', linewidth=1)
inset_ax.set_xlim(-0.2, 0.2)
inset_ax.tick_params(axis='both', which='major', labelsize=16)
inset_ax.set_xticks([-0.2, -0.1, 0.0, 0.1, 0.2])
inset_ax.set_yticks([])
inset_ax.set_title('Relative Differences', fontsize=19.5)
inset_ax.grid(axis='x')

plt.tight_layout()
plt.show()

### ETC Plots

In [ ]:
options = etc.ETCOptions(
    instrument = 'GIRMOS'
)
options.config_fov(
    fov = 2.0 * u.arcsec
)
options.config_spec(
    band = 'HK',
    R = 3000
)
options.config_target(
    spatial_type     = etc.SpatialType.SERSIC,
    sersic_Re        = 1.50*u.kpc,                      # Sersic effective radius (arcsec or kpc)
    sersic_n         = 1.0,                             # Sersic index
    sersic_q         = 0.5,                             # Sersic axis ratio
    sersic_pa        = 45.0*u.deg,                      # Sersic position angle

    spectral_type    = etc.SpectralType.GAUSSIAN,
    redshift         = 2.0,                             # Redshift is needed if you only specify a rest wavelength or set flux using lsfr/Av
    wvl_rest         = 0.656281 * u.micron,             # Rest wavelength of emission line in air
    flux             = 10*1e-16*u.erg/u.s/u.cm**2,         # Either integrated flux (eg. erg/s/cm**2) or surface brightness (eg. erg/s/cm**2/arcsec**2)
    dispersion       = 130*u.km/u.s,                    # Sets the width of the line
)
options.config_target(
    x = -0.25*u.arcsec,
    y = 0.3*u.arcsec,
    spatial_type=etc.SpatialType.POINT,
    spectral_type=etc.SpectralType.GAUSSIAN,
    flux = 1e-17*u.erg/u.s/u.cm**2,
    dispersion = 100*u.km/u.s,
    velocity_offset = 10*u.km/u.s
)
# options.config_target(
#     x = 0.5*u.arcsec,
#     y = 0.5*u.arcsec,
#     spatial_type=etc.SpatialType.POINT,
#     spectral_type=etc.SpectralType.GAUSSIAN,
#     flux = 1e-17*u.erg/u.s/u.cm**2,
#     dispersion = 100*u.km/u.s,
#     velocity_offset = -1*u.km/u.s
# )
# options.config_target(
#     x = 0.0*u.arcsec,
#     y = 0.0*u.arcsec,
#     spatial_type=etc.SpatialType.POINT,
#     spectral_type=etc.SpectralType.GAUSSIAN,
#     flux = 1e-17*u.erg/u.s/u.cm**2,
#     dispersion = 100*u.km/u.s,
#     velocity_offset = -200*u.km/u.s
# )
options.config_ao(
    psf_type = etc.PSFType.MOAO, 
    ngs = [
        {"r": 30.0, "theta":   0.0, "mag": 17.0},
        {"r": 30.0, "theta": 120.0, "mag": 17.0},
        {"r": 30.0, "theta": 240.0, "mag": 17.0},
    ],
)
options.config_atm(
    atm               = 'median',
    zenith_angle      = 20*u.deg,
    water_vapor       = 1.6*u.mm,
)
options.config_analysis(
    T_exp             = 600*u.s,
    N_exp             = 30
)
options.config_finish()

ref_path = '../../girmos-aosims/output/ref'
psf_data = etc.get_psf(options, ref_path=ref_path)
result = etc.compute(options, psf_data)

In [ ]:
psf = psf_data['psf']
pixel_scale = psf_data['pixel_scale']
pixel_scale_y = pixel_scale

psf_min  = psf[psf > 0].min()
psf_max  = psf.max()
psf      = np.log10(np.abs(psf/psf_max)+1e-10)  # Avoid log(0) issues
vmin     = max(-10, np.log10(np.abs(psf_min/psf_max)))
Ny, Nx   = psf.shape
xlim     = [-Nx//2*pixel_scale.to_value(u.mas), Nx//2*pixel_scale.to_value(u.mas)]
ylim     = [-Ny//2*pixel_scale_y.to_value(u.mas), Ny//2*pixel_scale_y.to_value(u.mas)]
x        = np.linspace(xlim[0], xlim[1], Nx)
y        = np.linspace(ylim[0], ylim[1], Ny)
levels   = [-3, -2.5, -2, -1.5, -1]

fig = plt.figure(figsize=(4, 4))
ax = fig.add_subplot(111)
im = plt.imshow(psf, extent=[xlim[0], xlim[1], ylim[0], ylim[1]], origin='lower', cmap='viridis', vmin=vmin, vmax=1.0)
cn = plt.contour(x, y, psf, levels=levels, colors='white', linewidths=0.5)

def _format_contour_label(x):
    s = f"{x:.2f}"
    if s.endswith("0"):
        s = f"{x:.1f}"
    return rf"{s}" if plt.rcParams["text.usetex"] else f"{s}"

plt.clabel(cn, cn.levels, inline=True, fmt=_format_contour_label, fontsize=8)
plt.xlabel('[mas]', fontsize=16)
plt.ylabel('[mas]', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
from astropy.visualization import PercentileInterval, ImageNormalize, AsinhStretch

xlim = np.array([-1, 1]) * options.fov.to_value(u.arcsec) / 2 * 1000
ylim = np.array([-1, 1]) * options.fov_y.to_value(u.arcsec) / 2 * 1000

interval = PercentileInterval(99.8)   # clip brightest ~0.2%
vmin, vmax = interval.get_limits(result.target_spatial_profile_hi)
norm_func = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(a=0.3), clip=True)

fig = plt.figure(figsize=(4, 4))
ax  = fig.add_subplot(111)
im = plt.imshow(result.target_spatial_profile_hi, origin='lower', extent=(xlim[0], xlim[1], ylim[0], ylim[1]), cmap='viridis', norm=norm_func)
plt.xlim(xlim)
plt.ylim(ylim)
plt.xlabel('[mas]', fontsize=16)
plt.ylabel('[mas]', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)
plt.grid(alpha=0.2)
plt.show()

fig = plt.figure(figsize=(4, 4))
ax  = fig.add_subplot(111)
im = plt.imshow(result.target_spatial_profile_convolved_hi, origin='lower', extent=(xlim[0], xlim[1], ylim[0], ylim[1]), cmap='viridis', norm=norm_func)
plt.xlim(xlim)
plt.ylim(ylim)
plt.xlabel('[mas]', fontsize=16)
plt.ylabel('[mas]', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
point_source_spatial_masks, _, point_source_target_ids = etc.create_point_source_spatial_masks(options, return_ids=True)
_, combined_spectral_mask = etc.create_source_spectral_masks(options, result, spatial_masks=point_source_spatial_masks, target_ids=point_source_target_ids, npix=3)

signal = result.signal
signal_noise = result.signal + result.noise
noise_var = result.noise_var

signal = etc.spatial_average_cube(signal, box=2)
signal_noise, noise_var = etc.spatial_average_cube(signal_noise, noise_var, box=2)

peak_signal, peak_noise_var = etc.get_peak_signal(signal_noise, noise_var, spectral_mask=combined_spectral_mask)
peak_signal_no_noise, peak_noise_var_no_noise = etc.get_peak_signal(signal, noise_var, spectral_mask=combined_spectral_mask)
peak_signal_no_smooth_no_noise, peak_noise_var_no_smooth_no_noise = etc.get_peak_signal(result.signal, result.noise_var, spectral_mask=combined_spectral_mask)

nPix = 4
radial_result = etc.snr_sersic_distances(options, peak_signal_no_smooth_no_noise, np.sqrt(peak_noise_var_no_smooth_no_noise), nPix=nPix, dist_Re=[0,1,2])

print(f"FOV: {options.fov.to_value(u.arcsec)}\" x {options.fov_y.to_value(u.arcsec)}\", R={options.R}, T={(options.T_exp*options.N_exp).to(u.hour):.1f}")
print(f"NGS: mag={','.join([f"{ngs['mag']:.1f}" for ngs in options.ngs])}, EE={psf_data['ee']:.3f}")
print(f"Target: r={options.fov_r:.1f}, theta={options.fov_theta:.1f}, z={options.targets[0].redshift}, Ha={np.atleast_1d(options.targets[0].wvl)[0].to(u.micron):.3f}, Flux={options.targets[0].flux:.1e}" + (f" + {options.targets[0].continuum:.1e} continuum" if options.targets[0].continuum is not None else ""))
print(f"Sersic: n={options.targets[0].sersic_n}, q={options.targets[0].sersic_q}, Re={options.targets[0].sersic_Re:.1f} ({options.targets[0].sersic_Re_local})")

for i, rr in enumerate(radial_result):
    print(f"SNR at {rr['label']} [{nPix}px]: {rr['snr']:.2f}")

point_colors = ["#FF00A6", "#00E5FF", "#FF7A00"]

fig = etc.plot_flux_fit_value(
    options, peak_signal, np.sqrt(peak_noise_var),
    target=options.targets[0], nRe=[1.0, 2.0],
    points=[rr["pixels"] for rr in radial_result], point_colors=point_colors,
    plot_snr=True, return_fig=True
)
ax = fig.axes[0]
fig.set_figwidth(8)
fig.set_figheight(8)
fig.delaxes(fig.axes[1])
plt.xlabel('spaxels', fontsize=16)
plt.ylabel('spaxels', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)
plt.xlim(10,30)
plt.ylim(10,30)
plt.grid(alpha=0.2)
plt.title(None)
plt.show()

### Files for Ivana

In [ ]:
_, _, coverage_edfn, _, _, asterism_moc_edfn = get_asterism_mask(asterisms, "../data/euclid/EuclidMOC_EDFN_rsd2024c_depth13_atLeast2visitsPlanned.fits", levels=14)
asterism_moc_edfn.save("../output/asterism-moc-edfn.fits", format='fits', overwrite=True)
print(f"EDFN Coverage: {coverage_edfn:.2%}")

_, _, coverage_cosmos, _, _, asterism_moc_cosmos = get_asterism_mask(asterisms, "../data/moc/CDS_II_284_cosmos.fits", levels=14)
asterism_moc_cosmos.save("../output/asterism-moc-cosmos.fits", format='fits', overwrite=True)
print(f"COSMOS Coverage: {coverage_cosmos:.2%}")